# Stage 6 — Data-efficiency seed-42 pilots

This restartable Colab notebook builds leakage-safe, group-aware nested 10/25/50/100% subsets from the saved Stage 1 training split. It runs the approved batch-32 smoke test on 10%, then seed-42 pilots for 10%, 25%, and 50%. The verified frozen Stage 3 seed-42 run is reused for 100% only after exact configuration and source-data checksum equivalence. Evaluation is validation-only; the held-out test split is never loaded. This notebook does not run the final three-seed sweep.


In [ ]:
%pip uninstall -y -q torchao
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" datasets pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

PAYLOAD_SHA256 = 'D92E409DA01BFB746D2F9C9CCCD6F3AB54D40E98A64145B7D243D7BCCF2B4879'
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
SOURCE_DIR = Path('/content/drive/MyDrive/RAFT/stage6_data_efficiency/source_snapshot')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in archive.namelist())
    archive.extractall(SOURCE_DIR)
print(f'Stage 6 source ready: {SOURCE_DIR}')
print(f'PAYLOAD_SHA256={PAYLOAD_SHA256}')


In [ ]:
import json
import os
import subprocess
import sys
import torch
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
os.environ['HF_HOME'] = '/content/huggingface_cache'
assert torch.cuda.is_available(), 'Select a free Colab GPU runtime (T4 is suitable) before continuing.'
required = [
    DRIVE_ROOT / 'stage1_scifact/processed/corpus.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_train.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_train.tsv',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_validation.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_validation.tsv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing restartable Drive inputs: {missing}'

rows = []
for fraction in (10, 25, 50, 100):
    config = json.loads((SOURCE_DIR / f'configs/stage6_data_efficiency_pilot_{fraction}.json').read_text())
    rows.append({
        'fraction_percent': fraction,
        'subset_seed': config['subset']['subset_seed'],
        'training_seed': config['training']['random_seed'],
        'training_queries': config['subset']['realized_query_count'],
        'training_pairs': config['subset']['training_pair_count'],
        'steps_per_epoch': config['subset']['steps_per_epoch'],
        'eval_steps': config['training']['evaluation_steps'],
        'train_now': config['execution']['train_this_pilot'],
    })
display(pd.DataFrame(rows))
print('GPU:', torch.cuda.get_device_name(0))
print('CONFIG AUDIT READY: validation-only, patience-2, best checkpoint by nDCG@10')


In [ ]:
completed = subprocess.run([
    sys.executable,
    str(SOURCE_DIR / 'src/run_stage6_pilots.py'),
    '--repo-dir', str(SOURCE_DIR),
    '--drive-root', str(DRIVE_ROOT),
], check=True)
print(completed)


In [ ]:
PILOT_RESULTS = DRIVE_ROOT / 'stage6_data_efficiency/pilot_results'
per_fraction = pd.read_csv(PILOT_RESULTS / 'stage6_pilot_per_fraction_validation.csv')
comparison = pd.read_csv(PILOT_RESULTS / 'stage6_pilot_comparison_validation.csv')
summary = json.loads((PILOT_RESULTS / 'stage6_pilot_summary.json').read_text())
equivalence = json.loads((PILOT_RESULTS / 'stage6_100_percent_equivalence.json').read_text())
smoke = json.loads((DRIVE_ROOT / 'stage6_data_efficiency/smoke_fraction10_seed42/results/stage6_smoke_result.json').read_text())
assert smoke['passed'] is True and smoke['config']['training']['batch_size'] == 32
assert summary['test_split_loaded'] is False and summary['final_three_seed_sweep_run'] is False
assert equivalence['passed'] is True and equivalence['test_split_loaded'] is False
print('SMOKE TEST: PASSED AT BATCH 32 ON THE 10% SUBSET')
print('100% STAGE 3 EXACT-EQUIVALENCE AUDIT: PASSED')
print('TEST SPLIT LOADED:', summary['test_split_loaded'])
print('FINAL THREE-SEED SWEEP RUN:', summary['final_three_seed_sweep_run'])
print()
print('STAGE 6 SEED-42 PILOTS:')
display(per_fraction)
print()
print('STAGE 2 BASELINES + STAGE 3/4/5 CONTEXT + STAGE 6 PILOTS:')
display(comparison)


In [ ]:
from pathlib import Path
import zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
PILOT_RESULTS = DRIVE_ROOT / 'stage6_data_efficiency/pilot_results'
RESULT_ARCHIVE = PILOT_RESULTS / 'stage6_pilot_results.zip'
result_names = [
    'stage6_pilot_per_fraction_validation.csv',
    'stage6_pilot_comparison_validation.csv',
    'stage6_pilot_summary.json',
    'stage6_100_percent_equivalence.json',
]
with zipfile.ZipFile(RESULT_ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in result_names:
        archive.write(PILOT_RESULTS / name, arcname=name)
    archive.write(
        DRIVE_ROOT / 'stage6_data_efficiency/subsets/stage6_subset_manifest.json',
        arcname='stage6_subset_manifest.json',
    )
    archive.write(
        DRIVE_ROOT / 'stage6_data_efficiency/smoke_fraction10_seed42/results/stage6_smoke_result.json',
        arcname='stage6_smoke_result.json',
    )
print(f'Attach this file in the Codex task: {RESULT_ARCHIVE}')
